In [1]:
from dotenv import load_dotenv
load_dotenv()

import langchain
#import langgraph
print(f"LangChain: {langchain.__version__}")
#print(f"LangGraph: {langgraph.__version__}")

LangChain: 1.3.11


In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gpt-oss:120b-cloud", temperature=0)

# ── Alternatively, use Groq ──────────────────────────────────────────────────
# from langchain_groq import ChatGroq
# llm = ChatGroq(model="llama-3.3-70b-versatile")
# ─────────────────────────────────────────────────────────────────────────────

print("LLM ready.")

LLM ready.


In [3]:
from langchain_core.tools import tool

# Reusing the familiar tools from previous lessons
@tool
def get_weather(city: str) -> str:
    """Returns the current weather for a given city."""
    return f"The weather in {city} is sunny with a high of 28°C."

@tool
def get_stock_price(ticker: str) -> str:
    """Returns the current stock price for a given ticker symbol.
    Use 'ZENSAR' for Zensar Technologies, 'GOOGL' for Google."""
    prices = {"ZENSAR": "464.00 INR", "GOOGL": "175.00 USD"}
    return prices.get(ticker.upper(), f"Unknown ticker: {ticker}")

print("Tools ready: get_weather, get_stock_price")

Tools ready: get_weather, get_stock_price


In [ ]:
# Part 1: The Problem — The Forgetful Agent
# Let's build the agent exactly as we did in previous lessons and show why it forgets

In [4]:
from langchain.agents import create_agent

# No checkpointer → no memory
agent_no_memory = create_agent(
    model=llm,
    tools=[get_weather, get_stock_price],
    system_prompt="You are a helpful assistant.",
)

print("Agent created (no memory).")

Agent created (no memory).


In [6]:
# Turn 1: Introduce ourselves
response1 = agent_no_memory.invoke({
    "messages": [("user", "Hi! My name is Sumit and I work at Zensar Technologies in Pune.")]
})

print("Turn 1 — Agent says:")
print(response1["messages"][-1].content)

Turn 1 — Agent says:
Nice to meet you, Sumit! 👋 I’m glad you reached out. How can I assist you today? Whether it’s something related to Zensar Technologies, a quick lookup, or anything else, just let me know!


In [ ]:
# Turn 2: Ask a follow-up — the agent should remember, but won't
response2 = agent_no_memory.invoke({
    "messages": [("user", "What is my name and where do I work?")]
})

print("Turn 2 — Agent says:")
print(response2["messages"][-1].content)
print()
print("☝️ The agent has NO idea — each invoke() starts a blank conversation.")     #Because it is stateless, it forgets everything from the previous turn.

Turn 2 — Agent says:
I’m not sure—I don’t have any information about you yet. Could you let me know your name and where you work, or give me any details you’d like me to use?

☝️ The agent has NO idea — each invoke() starts a blank conversation.


In [8]:
# The simplest fix: keep a Python list and append every message yourself.

# We can reuse the same agent — memory is not in the agent, it's in how we call it

conversation_history = []  # We manage this list

def chat(user_message: str) -> str:
    """Send a message and keep the full history for next time."""
    # Add the new user message to history
    conversation_history.append({"role": "user", "content": user_message})
    
    # Invoke with the FULL history every time
    response = agent_no_memory.invoke({"messages": conversation_history})
    
    # Extract the AI reply and store it too
    ai_reply = response["messages"][-1].content
    conversation_history.append({"role": "assistant", "content": ai_reply})
    
    return ai_reply

print("Manual chat() function ready.")

Manual chat() function ready.


In [9]:
print("=" * 60)
print("Turn 1:")
reply1 = chat("Hi! My name is Sumit and I work at Zensar in Kolkata.")
print(f"Agent: {reply1}")

print()
print("=" * 60)
print("Turn 2:")
reply2 = chat("What is the weather in Pune today?")
print(f"Agent: {reply2}")

print()
print("=" * 60)
print("Turn 3:")
reply3 = chat("What is my name and where do I work?")
print(f"Agent: {reply3}")
print()
print("✅ It remembers! Because we sent all 4 previous messages along with Turn 3.")

Turn 1:
Agent: Hello Sumit! 👋 It’s great to meet you. How can I assist you today?

Turn 2:
Agent: You’re in luck—Pune’s enjoying sunny skies today with a pleasant high of around 28 °C (≈ 82 °F). Let me know if you need anything else, such as a forecast for the rest of the week or any other details!

Turn 3:
Agent: Your name is **Sumit**, and you work at **Zensar** in **Kolkata**.

✅ It remembers! Because we sent all 4 previous messages along with Turn 3.


In [10]:
# Let's see what the history looks like after 3 turns

print("📋 Full conversation_history we're managing:")
print(f"Total messages stored: {len(conversation_history)}")
print()
for i, msg in enumerate(conversation_history):
    role = msg['role'].upper()
    content = msg['content'][:80] + '...' if len(msg['content']) > 80 else msg['content']
    print(f"  [{i+1}] {role}: {content}")

📋 Full conversation_history we're managing:
Total messages stored: 6

  [1] USER: Hi! My name is Sumit and I work at Zensar in Kolkata.
  [2] ASSISTANT: Hello Sumit! 👋 It’s great to meet you. How can I assist you today?
  [3] USER: What is the weather in Pune today?
  [4] ASSISTANT: You’re in luck—Pune’s enjoying sunny skies today with a pleasant high of around ...
  [5] USER: What is my name and where do I work?
  [6] ASSISTANT: Your name is **Sumit**, and you work at **Zensar** in **Kolkata**.


Approach 2 — MemorySaver (LangGraph In-Memory Checkpointer)

MemorySaver stores conversation state automatically in RAM.
You identify each conversation with a thread_id — just like a chat session ID.

thread_id = "user_123"  →  separate memory per user
thread_id = "user_456"  →  completely isolated conversation

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# Create a MemorySaver — stores state in RAM
memory = MemorySaver()     #people also use this variable as checkpointer, and later use in below line as checkpointer = checkpointer,

# Pass the checkpointer to create_agent
agent_with_memory = create_agent(
    model=llm,
    tools=[get_weather, get_stock_price],
    system_prompt="You are a helpful assistant.",
    checkpointer=memory,          # ← this is the only new line
)

print("Agent with MemorySaver ready.")

Agent with MemorySaver ready.


In [ ]:
# The thread_id ties all messages in a session together
# Think of it as a "conversation ID" or "session ID"
config = {"configurable": {"thread_id": "sumit_session_1"}}

print("=" * 60)
print("Turn 1:")
r1 = agent_with_memory.invoke(
    {"messages": [("user", "Hi! My name is Sharad and I work at Zensar in Pune.")]},
    config=config
)
print(f"Agent: {r1['messages'][-1].content}")

print()
print("=" * 60)
print("Turn 2:")
r2 = agent_with_memory.invoke(
    {"messages": [("user", "What is the weather in Pune today?")]},
    config=config
)
print(f"Agent: {r2['messages'][-1].content}")

print()
print("=" * 60)
print("Turn 3:")
r3 = agent_with_memory.invoke(
    {"messages": [("user", "What is my name and where do I work?")]},
    config=config
)
print(f"Agent: {r3['messages'][-1].content}")
print()
print("✅ Memory works — and we only passed ONE message per call!")

For Multiple User / Session

In [15]:
# Two different users - same agent, different thread_id

config_sumit = {"configurable": {"thread_id": "user_sumit"}}
config_diya = {"configurable": {"thread_id": "user_diya"}}

#sumit introduces himself

agent_with_memory.invoke(
    {"messages": [("user", "Hi! My name is Sumit and I work at Zensar in Pune.")]},
    config=config_sumit
)  

#diya introduces herself
agent_with_memory.invoke(
    {"messages": [("user", "Hi! My name is Diya and I am studying Medicine.")]},
    config=config_diya
)

# Ask Sumit session
r_sumit = agent_with_memory.invoke(
    {"messages": [("user", "What is my name and where do I work?")]},
    config=config_sumit
)
print(f"Sumit session —> Agent: {r_sumit['messages'][-1].content}")
print()

# Ask Diya session
r_diya = agent_with_memory.invoke(
    {"messages": [("user", "What is my name and what do I study?")]},
    config=config_diya
)
print(f"Diya session —> Agent: {r_diya['messages'][-1].content}")
print()

print("✅ Memory works for multiple users — each thread_id is a separate conversation.")


Sumit session —> Agent: Your name is **Sumit**, and you work at **Zensar** in **Pune**.

Diya session —> Agent: Your name is **Diya**, and you’re studying **medicine**.

✅ Memory works for multiple users — each thread_id is a separate conversation.


🔬 What Does MemorySaver Actually Store?

In [ ]:
# Retrieve the stored checkpoint for Sharad's session
config_to_inspect = {"configurable": {"thread_id": "user_sharad"}}
checkpoint = memory.get(config_to_inspect)

print("📦 Stored messages for 'user_sharad' thread:")
print()

if checkpoint:
    stored_messages = checkpoint["channel_values"].get("messages", [])
    for i, msg in enumerate(stored_messages):
        role = msg.__class__.__name__.replace("Message", "")
        content = msg.content[:80] + '...' if len(msg.content) > 80 else msg.content
        print(f"[{i+1}] {role}: {content}")
else:
    print("  No checkpoint found.")

print()
print("☝️ It's just the message list — same as our manual approach, but managed for us.")

📦 Stored messages for 'user_sharad' thread:

  No checkpoint found.

☝️ It's just the message list — same as our manual approach, but managed for us.
